In [1]:
import sys

# ============================================================
# Windows + Jupyter 下启动 stdio 类型 MCP（如 npx time-mcp）的兼容修复
# ------------------------------------------------------------
# 根因：mcp 库把 sys.stderr 作为子进程的 stderr。
#       Jupyter 里的 sys.stderr 是 ipykernel 的 OutStream，
#       它的 fileno() 会抛 UnsupportedOperation: fileno，
#       导致启动 MCP 子进程时报 "UnsupportedOperation: fileno"。
# 修复：给 OutStream 类打补丁，让 fileno() 返回真实 stderr 的 fd（通常是 2）。
#       这是类级别补丁，无论 mcp 在何时 import、errlog 何时被绑定都生效，
#       比 `sys.stderr = sys.__stderr__` 更可靠。
#
# 说明：Windows 上 Python 3.8+ 默认已使用 ProactorEventLoop（支持子进程），
#       无需再手动 set_event_loop_policy —— 该 API 在 Python 3.14 已弃用，
#       调用反而会产生 DeprecationWarning。
# ============================================================
if sys.platform == "win32" and "ipykernel" in sys.modules:
    from ipykernel.iostream import OutStream

    def _outstream_fileno(self):
        try:
            return sys.__stderr__.fileno()   # 返回真实 stderr 的 fd
        except Exception:
            return 2                          # 标准错误默认 fd

    OutStream.fileno = _outstream_fileno

# ------------------------------------------------------------
# 静默 time-mcp 的 stdout 噪音报错（可选，仅为让输出干净）
# ------------------------------------------------------------
# time-mcp 启动时会把 "Starting Time MCP server..." 用 process.stdout.write
# 写进 stdout，污染 MCP 的 JSON-RPC 通道。mcp 客户端解析这行会失败，但
# 只是记录一条日志后 continue，不影响工具加载。这里把这条特定日志静默掉。
# （time-mcp 1.0.0~1.0.6 全版本都有此问题且无开关，降版本无效）
import logging

class _McpNoiseFilter(logging.Filter):
    def filter(self, record):
        if "Failed to parse JSONRPC message from server" in record.getMessage():
            return False
        return True

logging.getLogger("mcp.client.stdio").addFilter(_McpNoiseFilter())

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
# ===================Step 1.初始化模型=====================
# 1.加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

import os
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 2.初始化模型
model = init_chat_model(
    model="kimi-k3", # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai", # 如果是Langchain不支持的模型，用openai的规范
    base_url=base_url,
    api_key=api_key
)
# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        # 原魔搭托管地址 https://mcp.api-inference.modelscope.net/50719920b4f24d/mcp
        # 已下线（HTTP 410 Gone），改成本地 stdio 启动，稳定且无需申请 Key。
        "12306-mcp": {
            "transport": "stdio",
            "command": "npx",
            "args": [
                "-y",
                "12306-mcp@0.3.10"
            ]
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()

# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.invoke调用Agent， 传入的Dict，里面有messages
response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下下周四从北京到杭州的高铁")]}
)
print(response)
# for message in response["messages"]:
#     message.pretty_print(message)


Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "D:\hm-insurance\agent-service\.venv\Lib\site-packages\mcp\client\stdio\__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
  File "D:\hm-insurance\agent-service\.venv\Lib\site-packages\pydantic\main.py", line 847, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        json_data, strict=strict, extra=extra, context=context, by_alias=by_alias, by_name=by_name
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='Starting Time MCP server...', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid
